In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes one argument:
- `n` is a natural number.

It returns the list
$$ [ 1, 2, 3,\cdots, n ]. $$

In [ ]:
let range (n : int) : int list =
  List.init n (fun i -> i + 1)

In [ ]:
range 10

# The Twin-Primes Problem

The notebook answers the following question: Could we decide the question whether there are infinitely many twin primes if the *halting problem* was solvable?

The function `divisors` takes one argument:
- `k` is a positive natural number.

It returns the list of all numbers from $1$ to $k$ that divide $k$ evenly.

In [ ]:
let divisors (k : int) : int list =
  range k |> List.filter (fun x -> k mod x = 0)

The function `is_prime` takes one argument:
- `k` is a positive natural number.

It returns `true` if $k$ is a prime number and `false` otherwise.  A number $k$ is prime if and only if it has exactly two divisors, namely $1$ and $k$.

In [ ]:
let is_prime (k : int) : bool =
  List.length (divisors k) = 2

The function `exists_prime_pair` takes one argument:
- `n` is a natural number.

It checks whether there is a pair $\langle p, p+2 \rangle$ of primes such that $p \geq n$.  In case a pair of primes is found, the first such pair is returned.
If there is no such pair, the function call `exists_prime_pair n` does not terminate.

**Implementation:** The search is done by the local recursive function `search`, which tests the numbers $p = n, n+1, n+2, \cdots$ one after the other.  Since the recursive call of `search` is a *tail call*, the recursion is translated into a loop by the compiler.

In [ ]:
let exists_prime_pair (n : int) : int * int =
  let rec search (p : int) : int * int =
    if is_prime p && is_prime (p + 2) then
      (p, p + 2)
    else
      search (p + 1)
  in
  search n

The function call `Sys.time ()` returns the processor time in seconds that has been used so far.  The difference of two such calls is the time needed by the computation in between.

In [ ]:
let start  = Sys.time () in
let (p, q) = exists_prime_pair 1_000_000 in
Printf.printf "(%d, %d) has been found in %.2f seconds.\n" p q (Sys.time () -. start)

To continue, we *assume* that there is a function `stops` that solves the halting problem.  This function takes two arguments:
- `f` is a function,
- `a` is an argument for the function `f`.

The function call `stops f a` returns
- `1` if the evaluation of `f a` terminates, and
- `0` if the evaluation of `f a` does not terminate.

Of course, such a function does not exist.  However, in OCaml every function has to be defined before it can be used.  Therefore, we define `stops` as a function that raises an exception.  The definition only serves to tell the compiler the type of `stops`: the type variables `'a` and `'b` specify that `stops` can be applied to any function `f` of type `'a -> 'b` and to any argument `a` of type `'a`.

In [ ]:
let stops (f : 'a -> 'b) (a : 'a) : int =
  failwith "The halting problem is not solvable!"

The function `find_counter_example` takes one argument:
- `n` is a natural number.

The function tries to find a natural number $b \geq n$ such that there are no twin primes bigger than $b$.  If such a number $b$ is found, it is returned.
In order to do so, it uses the function `stops` that solves the *halting problem*: if `stops exists_prime_pair b` returns `0`, then there is no pair of twin primes beyond `b`.

If there is no such number $b$, the function call `find_counter_example n` does not
terminate.

In [ ]:
let find_counter_example (n : int) : int =
  let rec search (b : int) : int =
    if stops exists_prime_pair b = 0 then  (* no prime pair beyond b *)
      b
    else
      search (b + 1)
  in
  search n

The function `is_twin_prime_conjecture_true` takes no arguments.
The function call `is_twin_prime_conjecture_true ()` returns `true`
if and only if the set of twin primes is infinite.  This is the case if and only if the function call
`find_counter_example 1` does not terminate.

In [ ]:
let is_twin_prime_conjecture_true () : bool =
  stops find_counter_example 1 = 0

Hence, if the halting problem were solvable, we could decide whether there are infinitely many twin primes.